# Análise de Churn de Clientes Bancários

Este notebook realiza uma análise exploratória de dados de clientes bancários com foco em entender padrões relacionados ao churn.

## Objetivo

Analisar o perfil dos clientes do banco, comparar clientes que cancelaram e não cancelaram o serviço, e identificar características associadas ao churn.

## Perguntas de Negócio

- Quais características são mais comuns entre clientes que cancelaram o serviço?
- Qual é o perfil demográfico geral dos clientes do banco?
- Existem diferenças de comportamento entre clientes da Alemanha, França e Espanha?
- Quais segmentos de clientes podem ser identificados na base?
- Quais variáveis parecem estar mais relacionadas ao churn?

## Etapas

1. Importação das bibliotecas
2. Carregamento dos dados
3. Exploração inicial
4. Limpeza e tratamento
5. Análise exploratória
6. Análise de churn
7. Análise por país
8. Visualizações
9. Insights finais
10. Conclusão

In [139]:
# 1-2. Importação das bibliotecas e Carregando os dados

import pandas as pd
import plotly.express as px

customer_info = pd.read_excel("../data/Bank_Churn_Messy.xlsx", sheet_name="Customer_Info")
account_info = pd.read_excel("../data/Bank_Churn_Messy.xlsx", sheet_name="Account_Info")

In [140]:
# 3. Exploração inicial

print(f"customer_info tem {customer_info.shape[0]} linhas e {customer_info.shape[1]} colunas.")
display(customer_info.head())
print(f"account_info tem {account_info.shape[0]} linhas e {account_info.shape[1]} colunas.")
display(account_info.head())


customer_info tem 10001 linhas e 8 colunas.


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary
0,15634602,Hargrave,619,FRA,Female,42.0,2,€101348.88
1,15647311,Hill,608,Spain,Female,41.0,1,€112542.58
2,15619304,Onio,502,French,Female,42.0,8,€113931.57
3,15701354,Boni,699,FRA,Female,39.0,1,€93826.63
4,15737888,Mitchell,850,Spain,Female,43.0,2,€79084.1


account_info tem 10002 linhas e 7 colunas.


,CustomerId,Balance,NumOfProducts,HasCrCard,Tenure,IsActiveMember,Exited
0,15634602,€0.0,1,Yes,2,Yes,1
1,15634602,€0.0,1,Yes,2,Yes,1
2,15647311,€83807.86,1,Yes,1,Yes,0
3,15619304,€159660.8,3,No,8,No,1
4,15701354,€0.0,2,No,1,No,0


In [141]:
# Informações e estatísticas descritivas das tabelas

print("customer_info")
customer_info.info()
display(customer_info.describe().round(2))

print("account_info")
account_info.info()
display(account_info.describe().round(2))

customer_info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001 entries, 0 to 10000
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       10001 non-null  int64  
 1   Surname          9998 non-null   object 
 2   CreditScore      10001 non-null  int64  
 3   Geography        10001 non-null  object 
 4   Gender           10001 non-null  object 
 5   Age              9998 non-null   float64
 6   Tenure           10001 non-null  int64  
 7   EstimatedSalary  10001 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 625.2+ KB


,CustomerId,CreditScore,Age,Tenure
count,10001.00,10001.00,9998.00,10001.00
mean,15690934.31,650.54,38.92,5.01
std,71935.31,96.66,10.49,2.89
min,15565701.00,350.00,18.00,0.00
25%,15628523.00,584.00,32.00,3.00
50%,15690733.00,652.00,37.00,5.00
75%,15753229.00,718.00,44.00,7.00
max,15815690.00,850.00,92.00,10.00


account_info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10002 entries, 0 to 10001
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CustomerId      10002 non-null  int64 
 1   Balance         10002 non-null  object
 2   NumOfProducts   10002 non-null  int64 
 3   HasCrCard       10002 non-null  object
 4   Tenure          10002 non-null  int64 
 5   IsActiveMember  10002 non-null  object
 6   Exited          10002 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 547.1+ KB


,CustomerId,NumOfProducts,Tenure,Exited
count,10002.00,10002.00,10002.00,10002.0
mean,15690928.68,1.53,5.01,0.2
std,71933.92,0.58,2.89,0.4
min,15565701.00,1.00,0.00,0.0
25%,15628524.75,1.00,3.00,0.0
50%,15690732.00,1.00,5.00,0.0
75%,15753225.50,2.00,7.00,0.0
max,15815690.00,4.00,10.00,1.0


In [142]:
# Padronizando os nomes das colunas
print(f"customer_info colunas antigas:\n {customer_info.columns}")
customer_info.columns = customer_info.columns.str.strip().str.lower().str.replace(" ","_")
print(f"customer_info colunas novas:\n {customer_info.columns}")

print(f"\naccount_info colunas antigas:\n {account_info.columns}")
account_info.columns = account_info.columns.str.strip().str.lower().str.replace(" ","_")
print(f"account_info colunas novas:\n {account_info.columns}")

customer_info colunas antigas:
 Index(['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age',
       'Tenure', 'EstimatedSalary'],
      dtype='object')
customer_info colunas novas:
 Index(['customerid', 'surname', 'creditscore', 'geography', 'gender', 'age',
       'tenure', 'estimatedsalary'],
      dtype='object')

account_info colunas antigas:
 Index(['CustomerId', 'Balance', 'NumOfProducts', 'HasCrCard', 'Tenure',
       'IsActiveMember', 'Exited'],
      dtype='object')
account_info colunas novas:
 Index(['customerid', 'balance', 'numofproducts', 'hascrcard', 'tenure',
       'isactivemember', 'exited'],
      dtype='object')


In [143]:
# 4. Limpeza e tratamento

# Somando duplicadas
print(f"Na tabela customer_info foram encontradas {customer_info.duplicated().sum()} linhas duplicadas.")
print(f"Na tabela account_info foram encontradas {account_info.duplicated().sum()} linhas duplicadas.")

Na tabela customer_info foram encontradas 1 linhas duplicadas.
Na tabela account_info foram encontradas 2 linhas duplicadas.


In [144]:
# Removendo duplicadas:

customer_info = customer_info.drop_duplicates()
print(f"Na tabela customer_info foram encontradas {customer_info.duplicated().sum()} linhas duplicadas.")

account_info = account_info.drop_duplicates()
print(f"Na tabela account_info foram encontradas {account_info.duplicated().sum()} linhas duplicadas.")

Na tabela customer_info foram encontradas 0 linhas duplicadas.
Na tabela account_info foram encontradas 0 linhas duplicadas.


In [145]:
# Identificando valores nulos
print("Tabela customer_info")
display(customer_info.isnull().sum())
print("Tabela account_info")
display(account_info.isnull().sum())

Tabela customer_info


customerid         0
surname            3
creditscore        0
geography          0
gender             0
age                3
tenure             0
estimatedsalary    0
dtype: int64

Tabela account_info


customerid        0
balance           0
numofproducts     0
hascrcard         0
tenure            0
isactivemember    0
exited            0
dtype: int64

In [146]:
# Foram encontrados apenas 3 valores nulos na coluna surname e 3 na age. 
# Como são valores pequenos, irei preencher na coluna idade com a mediana da coluna.

customer_info["age"] = customer_info["age"].fillna(customer_info["age"].median())
display(customer_info.isnull().sum())


customerid         0
surname            3
creditscore        0
geography          0
gender             0
age                0
tenure             0
estimatedsalary    0
dtype: int64

In [147]:
# E removemos a coluna surname por não possuir valor para nossa analise, sendo utilizada apenas para identificação dos clientes.
customer_info = customer_info.drop(columns=["surname"])
display(customer_info.isnull().sum())


customerid         0
creditscore        0
geography          0
gender             0
age                0
tenure             0
estimatedsalary    0
dtype: int64

In [148]:
# Unimos as tabelas para centralizar as informações dos clientes e facilitar a análise.
# Vou utilizar inner join para manter apenas clientes presentes nas duas tabelas, garantindo que a análise seja feita somente com registros que possuem informações completas.

df = customer_info.merge(account_info, on="customerid", how="inner")
print(f"O df tem {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()

O df tem 10000 linhas e 13 colunas.


,customerid,creditscore,geography,gender,age,tenure_x,estimatedsalary,balance,numofproducts,hascrcard,tenure_y,isactivemember,exited
0,15634602,619,FRA,Female,42.0,2,€101348.88,€0.0,1,Yes,2,Yes,1
1,15647311,608,Spain,Female,41.0,1,€112542.58,€83807.86,1,Yes,1,Yes,0
2,15619304,502,French,Female,42.0,8,€113931.57,€159660.8,3,No,8,No,1
3,15701354,699,FRA,Female,39.0,1,€93826.63,€0.0,2,No,1,No,0
4,15737888,850,Spain,Female,43.0,2,€79084.1,€125510.82,1,Yes,2,Yes,0


In [149]:
# Analisando as informações da nova tabela após a união

print(df.duplicated().sum())
print(df.info())

0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       10000 non-null  int64  
 1   creditscore      10000 non-null  int64  
 2   geography        10000 non-null  object 
 3   gender           10000 non-null  object 
 4   age              10000 non-null  float64
 5   tenure_x         10000 non-null  int64  
 6   estimatedsalary  10000 non-null  object 
 7   balance          10000 non-null  object 
 8   numofproducts    10000 non-null  int64  
 9   hascrcard        10000 non-null  object 
 10  tenure_y         10000 non-null  int64  
 11  isactivemember   10000 non-null  object 
 12  exited           10000 non-null  int64  
dtypes: float64(1), int64(6), object(6)
memory usage: 1015.8+ KB
None


In [150]:
# 5. Análise exploratória

# Criando uma nova coluna de churn com base na coluna exited para ficar mais facil a compreensão
df["churn"] = df["exited"]

# Contando a quantidade de cancelamentos
print(df["churn"].value_counts())
# Calculando a porcentagem de cancelamentos
print(df["churn"].value_counts(normalize=True))

churn
0    7963
1    2037
Name: count, dtype: int64
churn
0    0.7963
1    0.2037
Name: proportion, dtype: float64


In [151]:
# Pergunnta de negócio

#1 - Quais características são mais comuns entre clientes que cancelaram o serviço?

# Para responder essa pergunta irei analisar as informações da base como estatisticas gerais, distribuição por pais, genero e idade.
df.describe().round(2)

,customerid,creditscore,age,tenure_x,numofproducts,tenure_y,exited,churn
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.0,10000.0
mean,15690940.57,650.53,38.92,5.01,1.53,5.01,0.2,0.2
std,71936.19,96.65,10.49,2.89,0.58,2.89,0.4,0.4
min,15565701.00,350.00,18.00,0.00,1.00,0.00,0.0,0.0
25%,15628528.25,584.00,32.00,3.00,1.00,3.00,0.0,0.0
50%,15690738.00,652.00,37.00,5.00,1.00,5.00,0.0,0.0
75%,15753233.75,718.00,44.00,7.00,2.00,7.00,0.0,0.0
max,15815690.00,850.00,92.00,10.00,4.00,10.00,1.0,1.0


In [152]:
# Distribuição dos clientes por país
# Aqui meu objetivo é comparar onde a base de clientes fica mais centralizada.

print(df["geography"].value_counts())
print(df["geography"].value_counts(normalize=True).round(2))

geography
Germany    2509
Spain      2477
France     1741
French     1655
FRA        1618
Name: count, dtype: int64
geography
Germany    0.25
Spain      0.25
France     0.17
French     0.17
FRA        0.16
Name: proportion, dtype: float64


In [153]:
# Distribuição por gênero
# Entender a diferença de gênero que temos em nossa base

print(df["gender"].value_counts())
print(df["gender"].value_counts(normalize=True).round(2))

gender
Male      5457
Female    4543
Name: count, dtype: int64
gender
Male      0.55
Female    0.45
Name: proportion, dtype: float64


In [ ]:
# Criando um gráfico para analisar a dispensão de clientes por idade.

grafico_idade = px.histogram(df, x="age")
grafico_idade.show()

# import plotly.express as px

# for coluna in tabela:
#   grafico = px.histogram(tabela, x=coluna, color="Categoria")
#   grafico.show()
